# Community voting for uncertain snake sightings

A reviewer can vote for a species or stop at genus. Each reviewer has one active vote per sighting. The ballot returns species consensus, genus consensus, or pending.

[Van Horn et al. (2018), *Lean Multiclass Crowdsourcing*](https://openaccess.thecvf.com/content_cvpr_2018/CameraReady/1324.pdf), §3.1.2–3.1.3, motivates factoring a vote along the taxonomy and using only the levels the reviewer reported. [Wang, Lauha and Dunson (2024), *Bayesian modeling of multi-species labeling errors in ecological studies*](https://arxiv.org/html/2406.15844v1#S2.SS4), motivates allowing annotation accuracy to vary across reviewers. The model below adapts these ideas to one snake per sighting and uses uncalibrated role-level accuracies in the absence of reviewer history.

The latent label is a species in the project catalog, an unlisted species within a known genus, or an unlisted genus. The prior is uniform over genus options, then uniform over species options within each known genus. Model predictions are excluded from the prior for these low-confidence sightings. At each reported rank, a reviewer has the role-specific probability of giving the correct label and spreads the remaining probability uniformly across the other options. A species vote under the wrong genus is uniform over that reported genus's species options. The likelihood multiplies conditionally independent votes.

Sum species posterior mass within each genus to get the genus posterior. Select the most specific rank with a unique winner, posterior of at least 0.85, and at least 3 points of direct or rolled-up support for that winner. Unlisted alternatives can absorb posterior mass and keep a result pending.

In [ ]:
import sys
from collections import Counter
from math import fsum, isclose
from pathlib import Path

repo = Path.cwd()
if not (repo / "src").is_dir():
    repo = repo.parent
sys.path.insert(0, str(repo / "src"))

from serpentguard.community_voting import SPECIES_TO_GENUS, SightingBallot, consensus_label

## Voting scenarios

These checks use catalog taxa and synthetic ballots. The posterior measures evidence under the stated assumptions. Its empirical calibration requires labeled sightings with reviewer votes.

In [ ]:
REVIEWERS = {
    "citizen_1": "citizen",
    "citizen_2": "citizen",
    "citizen_3": "citizen",
    "rescuer_1": "verified_rescuer",
    "rescuer_2": "verified_rescuer",
    "expert_1": "expert",
}

scenarios = [
    ("No votes", [], "pending", None),
    ("Two citizens below minimum support", [
        ("citizen_1", "species", "Naja naja"),
        ("citizen_2", "species", "Naja naja"),
    ], "pending", None),
    ("Three agreeing citizens", [
        ("citizen_1", "species", "Naja naja"),
        ("citizen_2", "species", "Naja naja"),
        ("citizen_3", "species", "Naja naja"),
    ], "species_consensus", "Naja naja"),
    ("Same genus, different species", [
        ("citizen_1", "species", "Naja naja"),
        ("citizen_2", "species", "Naja kaouthia"),
        ("citizen_3", "genus", "Naja"),
    ], "genus_consensus", "Naja"),
    ("Direct genus vote from one expert", [
        ("expert_1", "genus", "Naja"),
    ], "genus_consensus", "Naja"),
    ("One expert species vote resolves only the genus", [
        ("expert_1", "species", "Naja naja"),
    ], "genus_consensus", "Naja"),
    ("Equal support for different genera", [
        ("rescuer_1", "genus", "Naja"),
        ("rescuer_2", "genus", "Ptyas"),
    ], "pending", None),
    ("Expert and rescuer outweigh one conflicting citizen", [
        ("expert_1", "species", "Naja naja"),
        ("rescuer_1", "species", "Naja naja"),
        ("citizen_1", "species", "Ptyas mucosa"),
    ], "species_consensus", "Naja naja"),
]

for name, votes, expected_status, expected_label in scenarios:
    ballot = SightingBallot(REVIEWERS)
    for vote in votes:
        ballot.cast_vote(*vote)
    outcome = ballot.result()
    assert (outcome["status"], outcome["label"]) == (expected_status, expected_label), (
        name, outcome["status"], outcome["label"]
    )
    assert outcome["reviewer_count"] == len(votes)
    assert isclose(fsum(outcome["genus_posterior"].values()), 1)
    total_species_mass = (
        fsum(outcome["species_posterior"].values())
        + fsum(outcome["unlisted_species_posterior"].values())
        + outcome["genus_posterior"][None]
    )
    assert isclose(total_species_mass, 1)
    for genus, genus_mass in outcome["genus_posterior"].items():
        if genus is not None:
            species_mass = fsum(
                probability for species, probability in outcome["species_posterior"].items()
                if SPECIES_TO_GENUS[species] == genus
            )
            assert isclose(species_mass + outcome["unlisted_species_posterior"][genus], genus_mass)

    confidence = outcome["posterior_confidence"]
    detail = "" if confidence is None else f" -> {outcome['label']} ({confidence:.3f})"
    print(f"{name}: {outcome['status']}{detail}")

In [ ]:
ballot = SightingBallot(REVIEWERS)
ballot.cast_vote("expert_1", "genus", "Naja")
original = ballot.result()
ballot.cast_vote("expert_1", "genus", "Naja")
assert ballot.result() == original

ballot.cast_vote("expert_1", "genus", "Ptyas")
changed = ballot.result()
assert changed["label"] == "Ptyas"
assert changed["reviewer_count"] == 1
assert changed["support_weight"]["genus"] == {"Ptyas": 3}
assert changed["support_weight"]["species"] == {}

invalid_votes = [
    ("unregistered", "genus", "Naja"),
    ("expert_1", "family", "Elapidae"),
    ("expert_1", "species", "Naja"),
    ("expert_1", "genus", None),
]
for vote in invalid_votes:
    try:
        ballot.cast_vote(*vote)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Accepted an invalid vote: {vote!r}")
assert ballot.result() == changed

# A genus-only vote preserves the relative probabilities within that genus.
assert isclose(original["species_posterior"]["Naja naja"],
               original["species_posterior"]["Naja kaouthia"])
assert isclose(original["species_posterior"]["Naja naja"],
               original["unlisted_species_posterior"]["Naja"])

assert consensus_label({"Naja": 0.85, "Ptyas": 0.15}, Counter(Naja=3)) == "Naja"
assert consensus_label({"Naja": 0.849, "Ptyas": 0.151}, Counter(Naja=3)) is None
assert consensus_label({"Naja": 0.95, "Ptyas": 0.05}, Counter(Naja=2)) is None
print("Vote replacement, repeat submissions, invalid votes, genus-only evidence,")
print("and threshold boundaries passed.")